In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [43]:
oil_price = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv')
ovx = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv')

oil_price['date'] = pd.to_datetime(oil_price['date'])
ovx['date'] = pd.to_datetime(ovx['date'])

df = oil_price[['date','Brent']].merge(ovx, on='date')

df = df.dropna()

C:\Users\JuanFranciscoPerez\AppData\Local\Temp\ipykernel_27252\3059274625.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  oil_price['date'] = pd.to_datetime(oil_price['date'])


In [44]:
df = df.sort_values('date').copy()
df['oil_ret_30d'] = np.log(df['Brent'].shift(-21) / df['Brent'])  # 21 trading days ≈ 30 calendar days
df = df.dropna(subset=['oil_ret_30d', 'OVXCLS'])


In [47]:
# %%
threshold = -0.10

df['oil_ret_1d'] = np.log(df['Brent'] / df['Brent'].shift(1))

# for each day, check: did a single-day drop of ≥10% happen in the next 21 trading days?
df['max_daily_drop_next21'] = df['oil_ret_1d'].rolling(21).min().shift(-21)
df['jump_next_month'] = df['max_daily_drop_next21'] < threshold

# %%
bins = [0, 30, 50, 70, 100, np.inf]
labels = ['<30', '30-50', '50-70', '70-100', '100+']
df['ovx_abs'] = pd.cut(df['OVXCLS'], bins=bins, labels=labels)

prob = df.dropna(subset=['ovx_abs', 'jump_next_month']).groupby('ovx_abs').agg(
    n=('jump_next_month', 'count'),
    prob_jump=('jump_next_month', 'mean'),
    n_jumps=('jump_next_month', 'sum'),
)

# conditional on a jump happening, how big is it?
# collect the actual daily returns that crossed the threshold
daily_jumps = df[df['oil_ret_1d'] < threshold].copy()
daily_jumps['ovx_abs'] = pd.cut(daily_jumps['OVXCLS'], bins=bins, labels=labels)

jump_sizes = daily_jumps.groupby('ovx_abs')['oil_ret_1d'].agg(
    mu_J='mean',
    sigma_J='std',
    n_events='count',
)

result = prob.join(jump_sizes)
print(f"P(at least one daily ≥{threshold*100}% oil drop in next 21 trading days) by OVX:")
print(result.round(4))

P(at least one daily ≥-10.0% oil drop in next 21 trading days) by OVX:
            n  prob_jump  n_jumps    mu_J  sigma_J  n_events
ovx_abs                                                     
<30      1251     0.0000        0     NaN      NaN       NaN
30-50    2583     0.0186       48     NaN      NaN       NaN
50-70     424     0.0307       13 -0.1204   0.0013       2.0
70-100    124     0.2097       26 -0.1237   0.0176       2.0
100+       41     0.6341       26 -0.2301   0.1486       4.0
